# 08. Calibration and the final test

**Where we are:** notebook 07 built quantile forecasts and found their intervals do not contain
the outcome as often as they should, especially with forecast weather.

**What this notebook does:**

1. Calibrates the intervals with conformal prediction, using only errors known at the time.
2. Runs the model **once** on the test window (June 2024 onwards), with forecast weather, exactly
   as it would have run live.
3. Compares it with NESO's day-ahead forecast and looks at where each does better.

**Background reading**
- [Romano, Patterson and Candès, Conformalized quantile regression (2019)](https://arxiv.org/abs/1905.03222)
- [Angelopoulos and Bates, A gentle introduction to conformal prediction (2021)](https://arxiv.org/abs/2107.07511)
- [Gneiting and Raftery, Strictly proper scoring rules (2007)](https://sites.stat.washington.edu/raftery/Research/PDF/Gneiting2007jasa.pdf), section 6.2 on the interval score


## Step 1. Set up and load


In [1]:
from pathlib import Path

import holidays
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROCESSED = Path("../data/processed")
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({
    "figure.dpi": 90,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e4e0",
    "lines.linewidth": 2,
})

features_actual = pd.read_parquet(PROCESSED / "features_actual.parquet")
features_forecast = pd.read_parquet(PROCESSED / "features_forecast.parquet")
validation = pd.read_parquet(PROCESSED / "validation_quantiles.parquet")
point_validation = pd.read_parquet(PROCESSED / "validation_forecasts.parquet")
point_validation = point_validation[point_validation["model"] == "LightGBM (actual weather)"]
validation = validation.merge(point_validation[["start_utc", "forecast"]].rename(columns={"forecast": "point"}),
                              on="start_utc")
bench = pd.read_parquet(PROCESSED / "neso_benchmark.parquet")

COLUMNS = ["slot", "weekday", "month", "day_of_year", "holiday", "christmas", "lag7", "lag14",
           "similar_day", "level24", "level_ratio", "solar_capacity", "temperature_2m",
           "temp_day_mean", "wind_speed_10m", "shortwave_radiation", "radiation_x_capacity"]
PARAMS = dict(n_estimators=600, learning_rate=0.05, num_leaves=63, min_child_samples=50,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, n_jobs=4, verbose=-1, random_state=0)
QUANTILES = [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]
QCOLS = [f"q{q}" for q in QUANTILES]
INTERVALS = [(0.25, 0.75), (0.1, 0.9), (0.05, 0.95)]


def block(slot):
    """Four parts of the local day: night, morning, middle of the day, evening."""
    hour = np.asarray(slot) / 2
    return np.select([hour < 6, hour < 10, hour < 16], ["night", "morning", "midday"], "evening")


def coverage(frame, low, high, lo_col=None, hi_col=None):
    y = frame["Demand_Outturn"]
    return ((y >= frame[lo_col or f"q{low}"]) & (y <= frame[hi_col or f"q{high}"])).mean()


## Step 2. Conformal calibration

**The idea:** look at how far outcomes fell *outside* each interval in the recent past, and
widen (or narrow) the interval by just enough that the right share of past outcomes would have
been inside. For an interval from quantile `lo` to `hi`, each past half-hour gets a score

`score = max(lo − outcome, outcome − hi)`

which is positive when the outcome fell outside and negative when it was inside. For a 90%
interval, the adjustment is the 90th percentile of the scores (with a small correction for
sample size), added to both ends. This is conformalized quantile regression (Romano et al., 2019).

**Why it helps here:** conformal methods promise the target coverage if the future errors behave
like the past ones. They do not need the quantile models to be right, only the recent errors to
be a fair sample of the next ones.

**Which past:** the previous six months of out-of-sample forecasts, rolled forward month by
month, so each month is calibrated only on errors known before it began.

**Three options compared:**
- conformal, one adjustment for the whole day;
- conformal, a separate adjustment for each of four parts of the day (night, morning, middle of
  the day, evening), since notebook 07 found coverage varies through the day;
- notebook 07's reference, the point forecast plus recent errors by half-hour, which is itself a
  simple conformal method built on the point model.

**How to choose:** the 90% *interval score* (Gneiting and Raftery, 2007): the width of the
interval, plus 20 times the distance by which the outcome falls outside it. It is a proper score,
so it rewards narrow intervals only when they still contain the outcome. Lower is better.


In [2]:
def conformal_shift(scores, nominal):
    n = len(scores)
    level = min(1.0, np.ceil((n + 1) * nominal) / n)
    return np.quantile(scores, level, method="higher")


def calibrate(frame, history, by_block):
    """Add calibrated interval ends to frame, using scores from history."""
    out = frame.copy()
    groups_now = block(out["slot"]) if by_block else np.full(len(out), "all")
    groups_past = block(history["slot"]) if by_block else np.full(len(history), "all")
    for low, high in INTERVALS:
        score = np.maximum(history[f"q{low}"] - history["Demand_Outturn"],
                           history["Demand_Outturn"] - history[f"q{high}"])
        shift = {g: conformal_shift(score[groups_past == g], high - low) for g in np.unique(groups_past)}
        delta = pd.Series(groups_now, index=out.index).map(shift)
        out[f"c{low}"] = out[f"q{low}"] - delta
        out[f"c{high}"] = out[f"q{high}"] + delta
    return out


def reference(frame, history):
    """Point forecast plus quantiles of recent errors, separately for each half-hour of the day."""
    out = frame.copy()
    errors = (history["Demand_Outturn"] - history["point"]).groupby(history["slot"])
    for low, high in INTERVALS:
        out[f"c{low}"] = out["point"] + out["slot"].map(errors.quantile(low))
        out[f"c{high}"] = out["point"] + out["slot"].map(errors.quantile(high))
    return out


def interval_score(frame, low=0.05, high=0.95, lo_col="c0.05", hi_col="c0.95"):
    y, lo, hi = frame["Demand_Outturn"], frame[lo_col], frame[hi_col]
    alpha = 1 - (high - low)
    return ((hi - lo) + 2 / alpha * ((lo - y).clip(lower=0) + (y - hi).clip(lower=0))).mean()


METHODS = {
    "conformal, global": lambda frame, history: calibrate(frame, history, by_block=False),
    "conformal, by time of day": lambda frame, history: calibrate(frame, history, by_block=True),
    "point + recent errors": reference,
}
folds = sorted(validation["fold"].unique())
calibrated = {}
for name, method in METHODS.items():
    parts = []
    for i, fold in enumerate(folds[6:], start=6):
        history = validation[validation["fold"].isin(folds[i - 6:i])]
        parts.append(method(validation[validation["fold"] == fold], history))
    calibrated[name] = pd.concat(parts, ignore_index=True)

raw = validation[validation["fold"].isin(folds[6:])]
rows = {"raw quantiles": {**{f"{h - l:.0%} coverage": coverage(raw, l, h) for l, h in INTERVALS},
                          "90% width (MW)": (raw["q0.95"] - raw["q0.05"]).mean(),
                          "90% interval score (MW)": interval_score(raw, lo_col="q0.05", hi_col="q0.95")}}
for name, frame in calibrated.items():
    rows[name] = {**{f"{h - l:.0%} coverage": coverage(frame, l, h, f"c{l}", f"c{h}") for l, h in INTERVALS},
                  "90% width (MW)": (frame["c0.95"] - frame["c0.05"]).mean(),
                  "90% interval score (MW)": interval_score(frame)}
comparison = pd.DataFrame(rows).T
comparison.round(3)


,50% coverage,80% coverage,90% coverage,90% width (MW),90% interval score (MW)
raw quantiles,0.292,0.567,0.735,2633.974,5929.285
"conformal, global",0.498,0.794,0.892,3798.801,5114.004
"conformal, by time of day",0.498,0.794,0.892,3824.049,5047.899
point + recent errors,0.479,0.766,0.871,3158.036,4765.168


In [3]:
gap = {}
for name, frame in [("raw", raw), *calibrated.items()]:
    lo, hi = ("q0.05", "q0.95") if name == "raw" else ("c0.05", "c0.95")
    inside = (frame["Demand_Outturn"] >= frame[lo]) & (frame["Demand_Outturn"] <= frame[hi])
    gap[name] = inside.groupby(block(frame["slot"])).mean()
by_part = pd.DataFrame(gap).round(3)
by_part.loc["mean width of 90% interval (MW)"] = [
    (raw["q0.95"] - raw["q0.05"]).mean(),
    *[(f["c0.95"] - f["c0.05"]).mean() for f in calibrated.values()]]
by_part.round(3)


,raw,"conformal, global","conformal, by time of day",point + recent errors
evening,0.740,0.900,0.891,0.874
midday,0.700,0.833,0.891,0.875
morning,0.765,0.902,0.891,0.863
night,0.743,0.932,0.894,0.871
mean width of 90% interval (MW),2633.974,3798.801,3824.049,3158.036


In [4]:
METHOD = comparison.drop(index="raw quantiles")["90% interval score (MW)"].idxmin()
print("chosen method:", METHOD)


chosen method: point + recent errors


**Found (validation, December 2022 to May 2024):**

- Conformal calibration fixes the quantile models' coverage: from 29%, 57% and 74% to 50%, 79% and
  89% for the 50%, 80% and 90% intervals, close to target.
- Calibrating separately by time of day evens out coverage through the day (89% in every part,
  against 83% at midday with one global adjustment) at almost no extra width.
- **But the reference still wins on the interval score:** 4,765 MW against 5,048 MW for the best
  conformal option. Its 90% interval is narrower (3,158 MW against 3,824 MW) at slightly lower
  coverage (87%), and the score judges that a better trade.
- The reason is the same as in notebook 07: the reference is built around the point forecast's
  out-of-sample errors. The quantile models need a large conformal correction to reach the same
  coverage, and that correction widens every interval equally, including the ones that were
  already wide enough.

**Decision:** the test window uses the point forecast plus recent forecast-weather errors, chosen
here on validation data before the test is run.


## Step 3. The final test run

**What:** for each month from June 2024 to November 2026, train the point model (notebook 06) and
the seven quantile models (notebook 07) on everything known before the month, then forecast the
month using **day-ahead weather forecasts**. This is the first and only time the model sees the
test window.

**The rules, as in a live system:**
- Training uses ERA5 weather, which is published about five days late, so training data stops
  seven days before each month starts.
- Forecasts use `_previous_day2` weather, `INDO` lags up to 08:00 UTC the day before, and solar
  capacity from four weeks earlier (notebook 04).
- The model is refitted monthly; the settings are fixed from notebooks 06 and 07, chosen on
  validation data only.


In [5]:
covered = features_forecast.groupby("Date")["shortwave_radiation"].apply(lambda s: s.notna().all())
last_outturn = pd.to_datetime(features_forecast.loc[features_forecast["Demand_Outturn"].notna(), "Date"]).max()
SPRING_FOLDS = [(pd.Timestamp("2024-03-08"), pd.Timestamp("2024-03-31")),
                (pd.Timestamp("2024-04-01"), pd.Timestamp("2024-04-30")),
                (pd.Timestamp("2024-05-01"), pd.Timestamp("2024-05-31"))]
TEST_FOLDS = [(s, min(s + pd.offsets.MonthEnd(0), last_outturn))
              for s in pd.date_range("2024-06-01", last_outturn, freq="MS")]
train_dates = pd.to_datetime(features_actual["Date"])
test_dates = pd.to_datetime(features_forecast["Date"])

parts = []
for start, end in SPRING_FOLDS + TEST_FOLDS:
    train = features_actual[(train_dates <= start - pd.Timedelta(days=7)) & features_actual["ND"].notna()]
    target = train["ND"] - train["level24"]
    test = features_forecast[test_dates.between(start, end) & features_forecast["Demand_Outturn"].notna()
                             & features_forecast["Date"].map(covered).fillna(False).astype(bool)]
    out = test[["start_utc", "Date", "slot", "Demand_Forecast", "Demand_Outturn"]].assign(
        fold=f"{start:%Y-%m}", spring=start < pd.Timestamp("2024-06-01"))
    out["point"] = lgb.LGBMRegressor(**PARAMS).fit(train[COLUMNS], target).predict(test[COLUMNS]) + test["level24"]
    for q in QUANTILES:
        model = lgb.LGBMRegressor(objective="quantile", alpha=q, **PARAMS).fit(train[COLUMNS], target)
        out[f"q{q}"] = model.predict(test[COLUMNS]) + test["level24"]
    parts.append(out)
    print(f"{start:%Y-%m}: trained on {len(train):,} rows, forecast {len(out):,}")

runs = pd.concat(parts, ignore_index=True)
runs[QCOLS] = np.sort(runs[QCOLS].to_numpy(), axis=1)
print(f"{len(runs):,} half-hours forecast, of which {int(runs['spring'].sum()):,} in spring 2024 (calibration only)")


2024-03: trained on 90,576 rows, forecast 1,150


2024-04: trained on 91,728 rows, forecast 1,440


2024-05: trained on 93,166 rows, forecast 1,488


2024-06: trained on 94,654 rows, forecast 1,440


2024-07: trained on 96,094 rows, forecast 1,488


2024-08: trained on 97,582 rows, forecast 1,488


... (23 outputs left out)

2026-08: trained on 132,622 rows, forecast 1,488


2026-09: trained on 134,110 rows, forecast 1,440


2026-10: trained on 135,550 rows, forecast 1,490


2026-11: trained on 136,750 rows, forecast 432
46,896 half-hours forecast, of which 4,078 in spring 2024 (calibration only)


**Calibrating the test intervals:** the validation errors in Step 2 came from actual weather,
so they understate the errors with forecast weather. For the test, each month is calibrated on
forecast-weather errors only, with the method chosen in Step 2: spring 2024 (forecast above, before
the test window) to begin with, then each completed test month as it becomes known, keeping the
latest six months. Only days at least two days before the month starts are used, since outturn is
published after the day.


In [6]:
run_dates = pd.to_datetime(runs["Date"])
parts = []
for start, end in TEST_FOLDS:
    history = runs[(run_dates <= start - pd.Timedelta(days=2)) & (run_dates > start - pd.DateOffset(months=6))]
    parts.append(METHODS[METHOD](runs[runs["fold"] == f"{start:%Y-%m}"], history))
test_run = pd.concat(parts, ignore_index=True)
test_run.to_parquet(PROCESSED / "test_forecasts.parquet", index=False)
test_run[["Date", "Demand_Outturn", "Demand_Forecast", "point", "c0.05", "q0.5", "c0.95"]].head(3).round(0)


,Date,Demand_Outturn,Demand_Forecast,point,c0.05,q0.5,c0.95
0,2024-06-01,20626.0,20040.0,20611.0,19356.0,20635.0,21786.0
1,2024-06-01,20072.0,19218.0,20190.0,18967.0,20174.0,21417.0
2,2024-06-01,19595.0,18710.0,19745.0,18533.0,19780.0,20986.0
